In [13]:
import pandas as pd
import re
import jieba
import warnings
warnings.filterwarnings('ignore')

# 基础过滤条件
HIGH_VOTE_THRESHOLD = 100
MIN_COMMENT_LENGTH = 3

# 全局通用停用词（适用于所有电影的无意义词）
GLOBAL_STOPWORDS = {
    "的", "了", "在", "是", "我", "我们", "有", "和", "就", "不", "人",
    "都", "一", "一个", "上", "也", "很", "到", "说", "要", "去",
    "你", "会", "着", "没有", "看", "好", "这", "那", "但", "而",
    "与", "或", "被", "把", "让", "从", "对", "为", "以", "及",
    "等", "中", "之", "其", "于", "如", "则", "虽", "因", "所",
    "啊", "吧", "呢", "嗯", "哦", "哈", "嘛", "呀", "哇", "哎",
    "这个", "那个", "这种", "那种", "这部", "那部", "真的", "感觉",
    "觉得", "还是", "已经", "可以", "应该", "只是", "但是", "所以",
    "因为", "虽然", "如果", "不过", "然后", "还有", "而且", "并且",
    "看完", "看了", "看过", "自己", "什么", "怎么", "这么", "就是", 
    "这样", "作为", "前面", "最后", "开始", "时候", "一次", "不是", 
    "看到", "听说", "认为", "这是", "才能", "为什么", "非常", "现在", 
    "部分", "除了", "其他", "那么", "对于", "完全", "一点", "明明", 
    "简直", "本来", "知道", "想着", "可能", "只有", "电影", "影片", "片子", "观众", "一部"
}


每部电影的配置字典

In [14]:
MOVIE_CONFIGS = {
    "飞驰人生3": {
        "input_file": "../1.爬虫/飞驰人生3.csv",
        "output_file": "飞驰人生3_cleaned.csv",
        "custom_words": [
            "飞驰人生", "张驰", "沈腾", "韩寒", "孙宇强", "记星",
            "林臻东", "叶经理", "魏翔", "沙溢", "黄景瑜", "尹正",
            "沐尘100", "拉力赛", "赛车手", "换胎", "路书", "戈壁",
            "砂石路", "柏油路", "IMAX", "AI", "中速天梯", "巴音布鲁克"
        ]
    },
    "惊蛰无声": {
        "input_file": "../1.爬虫/惊蛰无声.csv",
        "output_file": "惊蛰无声_cleaned.csv",
        "custom_words": [
            "惊蛰无声", "张译", "宋佳", "朱一龙", "杨幂", "易烊千玺", 
            "四字", "雷佳音", "姚安娜", "刘诗诗", "国安", "张艺谋", "国师", 
            "老谋子", "黄队", "黄凯", "严迪", "严队", "白帆", "赵虹", "小玉", 
            "惊蛰行动", "惊蛰", "无间道", "谍战片", "谍战", "悬崖之上"
        ]
    },
    "镖人：风起大漠": {
        "input_file": "../1.爬虫/镖人_风起大漠.csv",
        "output_file": "镖人_风起大漠_cleaned.csv",
        "custom_words": [
            "镖人", "风起大漠", "吴京", "谢霆锋", "李连杰", "张晋", "于适", 
            "陈丽君", "梁家辉", "此沙", "惠英红", "袁和平", "刀马", "阿育娅", 
            "阿妮", "谛听", "知世郎", "燕子娘", "小七", "竖", "老莫", "左骁骑", 
            "大沙暴", "武侠片", "漫改", "打戏", "文戏", "武戏", "八爷", "袁导", 
            "玉面鬼", "裴行俨", "李云霄", "陈十九", "动作片"
        ]
    }
}


辅助函数：正则清理 + 时段提取

In [15]:
def clean_text_regex(text):
    """正则清洗文本"""
    # 去除 emoji 和 特殊符号
    emoji_pattern = re.compile(
        "[\U00010000-\U0010ffff\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub("", str(text))
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9，。！？、；：\u201c\u201d\u2018\u2019（）【】…—\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def get_time_period(dt):
    """时段提取"""
    if pd.isna(dt): return "未知"
    hour = dt.hour
    if 6 <= hour < 12: return "上午"
    elif 12 <= hour < 18: return "下午"
    elif 18 <= hour < 24: return "晚上"
    else: return "深夜"

In [16]:
for movie_key, config in MOVIE_CONFIGS.items():
    """处理单部电影的完整流水线"""
    print(f"\n{'='*40}")
    print(f"🚀 开始处理项目: {movie_key}")
    print(f"{'='*40}")
    
    # 1. 动态加载该电影的专属配置
    for word in config["custom_words"]:
        jieba.add_word(word)
    
    # 2. 读取数据
    try:
        df = pd.read_csv(config["input_file"], encoding='utf-8-sig')
        original_count = len(df)
        print(f"✅ 数据加载成功，初始数据量：{original_count} 条")
    except FileNotFoundError:
        print(f"❌ 找不到文件 {config['input_file']}，跳过该项目。")
        continue
        
    # 3. 基础清洗去重
    df = df.drop_duplicates(subset=["comment_text", "created_at"], keep="first")
    df = df[df["comment_text"].notna() & (df["comment_text"].str.strip() != "")]
    df = df[df["rating"].notna()]
    df = df[df["created_at"].notna() & (df["created_at"].str.strip() != "")]
    df = df[df["comment_length"] >= MIN_COMMENT_LENGTH]
    print(f"✅ 基础过滤完成，剩余：{len(df)} 条")
    
    # 4. 文本清洗与分词
    df["comment_text"] = df["comment_text"].apply(clean_text_regex)
    df["comment_length"] = df["comment_text"].str.len()
    df = df[df["comment_length"] >= MIN_COMMENT_LENGTH]
    
    def tokenize(text):
        words = jieba.cut(text, cut_all=False)
        return " ".join([w for w in words if w.strip() and w not in GLOBAL_STOPWORDS and len(w) > 1])
    
    df["tokens"] = df["comment_text"].apply(tokenize)
    print(f"✅ 文本清洗与专属词典分词完成")
    
    # 5. 衍生字段构造
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    df["comment_date"] = df["created_at"].dt.date
    df["time_period"] = df["created_at"].apply(get_time_period)
    df["is_high_vote"] = (df["votes_count"] > HIGH_VOTE_THRESHOLD).astype(int)
    
    # 6. 保存输出
    fieldnames = [
        "movie_name", "comment_type", "comment_text", "tokens",
        "rating", "votes_count", "is_high_vote",
        "created_at", "comment_date", "time_period",
        "comment_length", "source_page",
    ]
    df = df[fieldnames].reset_index(drop=True)
    df.to_csv(config["output_file"], index=False, encoding='utf-8-sig')
    
    print(f"🎉 处理完毕！已导出至 {config['output_file']}")
    print(f"   最终保留: {len(df)} 条 (剔除 {original_count - len(df)} 条)")


🚀 开始处理项目: 飞驰人生3
✅ 数据加载成功，初始数据量：1200 条
✅ 基础过滤完成，剩余：1190 条
✅ 文本清洗与专属词典分词完成
🎉 处理完毕！已导出至 飞驰人生3_cleaned.csv
   最终保留: 1186 条 (剔除 14 条)

🚀 开始处理项目: 惊蛰无声
✅ 数据加载成功，初始数据量：1199 条
✅ 基础过滤完成，剩余：1194 条
✅ 文本清洗与专属词典分词完成
🎉 处理完毕！已导出至 惊蛰无声_cleaned.csv
   最终保留: 1193 条 (剔除 6 条)

🚀 开始处理项目: 镖人：风起大漠
✅ 数据加载成功，初始数据量：1200 条
✅ 基础过滤完成，剩余：1194 条
✅ 文本清洗与专属词典分词完成
🎉 处理完毕！已导出至 镖人_风起大漠_cleaned.csv
   最终保留: 1194 条 (剔除 6 条)
